# Lab 5E: Partially Observable Games and Limits of Game Search

## Learning goals
By the end of this lab, students can:
- explain why imperfect information changes the game tree into information sets and beliefs;
- compute a simple Bayesian belief update in a bluffing card game;
- visualize hidden states that look identical to a player;
- demonstrate the horizon effect from heuristic cutoff search;
- demonstrate why forward pruning can be dangerous;
- connect these limitations to exponential growth in game trees.

Source note: Chapter 5 introduces games with imperfect information and highlights practical limitations such as cutoff errors, horizon effects, and risky forward pruning.

## Part A: One-card bluffing game

Deck: `J`, `Q`, `K`. Each player gets one private card.

1. Player 1 may **Check** or **Bet**.
2. If Player 1 checks, the higher card wins 1 chip.
3. If Player 1 bets, Player 2 may **Fold** or **Call**.
4. If Player 2 folds, Player 1 wins 1 chip. If Player 2 calls, the higher card wins 2 chips.

The key idea: after seeing a bet, Player 2 does not know Player 1's card. Player 2 groups several hidden states into one **information set** and acts from a belief distribution.

In [ ]:
# Run this cell first.
# %pip install matplotlib ipywidgets

import math
import itertools

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, FancyBboxPatch

from IPython.display import display, HTML, clear_output
try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    WIDGETS_AVAILABLE = False
    print("ipywidgets is not available. Static displays will still work.")

CARDS = ("J", "Q", "K")
RANK = {"J": 1, "Q": 2, "K": 3}


def showdown_payoff_p1(p1_card, p2_card, pot_units=1):
    return pot_units if RANK[p1_card] > RANK[p2_card] else -pot_units


def posterior_p1_card(p2_card, observed_action, bet_probs):
    """Bayes update for P2's belief about P1's card after observing Bet or Check."""
    possible = [c for c in CARDS if c != p2_card]
    weights = {}
    for c in possible:
        likelihood = bet_probs[c] if observed_action == "Bet" else (1 - bet_probs[c])
        weights[c] = likelihood * (1 / len(possible))
    z = sum(weights.values())
    if z == 0:
        return {c: 1/len(possible) for c in possible}
    return {c: w/z for c, w in weights.items()}


def p2_call_ev(p2_card, belief):
    # EV for Player 2 if calling a bet. Player 2 wins +2 if its card is higher, else -2.
    ev = 0
    for p1_card, prob in belief.items():
        ev += prob * (2 if RANK[p2_card] > RANK[p1_card] else -2)
    return ev


def p2_best_response(p2_card, belief):
    call_ev = p2_call_ev(p2_card, belief)
    fold_ev = -1
    return ("Call" if call_ev >= fold_ev else "Fold"), call_ev, fold_ev


def expected_value_p1_policy(bet_probs):
    """Expected value to P1 when P2 best-responds to a bet using beliefs."""
    total = 0
    deals = [(a, b) for a in CARDS for b in CARDS if a != b]
    for p1_card, p2_card in deals:
        p_bet = bet_probs[p1_card]
        check_payoff = showdown_payoff_p1(p1_card, p2_card, pot_units=1)
        belief = posterior_p1_card(p2_card, "Bet", bet_probs)
        response, _, _ = p2_best_response(p2_card, belief)
        if response == "Fold":
            bet_payoff = 1
        else:
            bet_payoff = showdown_payoff_p1(p1_card, p2_card, pot_units=2)
        total += (1 - p_bet) * check_payoff + p_bet * bet_payoff
    return total / len(deals)

DEFAULT_BET_PROBS = {"J": 0.20, "Q": 0.55, "K": 0.90}
print("Default policy EV for P1:", round(expected_value_p1_policy(DEFAULT_BET_PROBS), 3))

In [ ]:
def draw_card(ax, x, y, label, face_down=False, width=0.8, height=1.1):
    face = "#2b2d42" if face_down else "white"
    edge = "black"
    ax.add_patch(FancyBboxPatch((x-width/2, y-height/2), width, height,
                                boxstyle="round,pad=0.04", facecolor=face, edgecolor=edge, lw=1.5))
    ax.text(x, y, "?" if face_down else label, ha="center", va="center", fontsize=20,
            color="white" if face_down else "black", fontweight="bold")


def draw_belief_view(p2_card="Q", observed_action="Bet", bet_probs=None):
    if bet_probs is None:
        bet_probs = DEFAULT_BET_PROBS
    belief = posterior_p1_card(p2_card, observed_action, bet_probs)
    response, call_ev, fold_ev = p2_best_response(p2_card, belief)
    fig = plt.figure(figsize=(13, 4.8))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.1, 1.2, 1.4])
    ax_cards = fig.add_subplot(gs[0, 0])
    ax_bars = fig.add_subplot(gs[0, 1])
    ax_tree = fig.add_subplot(gs[0, 2])

    ax_cards.axis("off")
    ax_cards.set_xlim(0, 4)
    ax_cards.set_ylim(0, 3)
    ax_cards.set_title("What Player 2 sees")
    draw_card(ax_cards, 1, 1.7, p2_card, face_down=False)
    ax_cards.text(1, 0.8, "P2 private card", ha="center")
    draw_card(ax_cards, 3, 1.7, "?", face_down=True)
    ax_cards.text(3, 0.8, "P1 hidden card", ha="center")
    ax_cards.text(2, 2.65, f"Observed action: {observed_action}", ha="center", fontsize=12, fontweight="bold")

    cards = list(belief.keys())
    probs = [belief[c] for c in cards]
    ax_bars.bar(cards, probs, color="#8ecae6", edgecolor="black")
    ax_bars.set_ylim(0, 1)
    ax_bars.set_ylabel("P2 belief probability")
    ax_bars.set_title("Bayesian belief over P1 card")
    for i, p in enumerate(probs):
        ax_bars.text(i, p + 0.03, f"{p:.2f}", ha="center")
    ax_bars.text(0.5, -0.28, f"P2 call EV = {call_ev:.2f}; fold EV = {fold_ev:.2f}; best response = {response}",
                 transform=ax_bars.transAxes, ha="center", fontsize=9)

    draw_information_set_tree(ax_tree, p2_card, observed_action, belief, response)
    fig.tight_layout()
    plt.show()


def draw_information_set_tree(ax, p2_card, observed_action, belief, response):
    ax.axis("off")
    ax.set_title("Information set after observed action")
    ax.set_xlim(-1, 5)
    ax.set_ylim(-0.5, 3.5)
    ax.text(2, 3.2, "P2 decision nodes share one information set", ha="center", fontsize=10)
    possible = list(belief.keys())
    xs = [1.2, 3.2]
    for x, p1_card in zip(xs, possible):
        ax.plot([2.2, x], [2.7, 1.9], color="#444444")
        ax.text((2.2+x)/2, 2.35, f"P1={p1_card}\nprob={belief[p1_card]:.2f}", ha="center", fontsize=8)
        ax.add_patch(Rectangle((x-0.6, 1.45), 1.2, 0.7, facecolor="#d7e9ff", edgecolor="black"))
        ax.text(x, 1.8, f"P2 node\nP2={p2_card}", ha="center", va="center", fontsize=8)
        ax.plot([x, x-0.4], [1.45, 0.7], color="#444444")
        ax.plot([x, x+0.4], [1.45, 0.7], color="#444444")
        ax.text(x-0.55, 0.55, "Fold", fontsize=8, ha="center")
        ax.text(x+0.55, 0.55, "Call", fontsize=8, ha="center")
    # dashed information-set box
    ax.add_patch(Rectangle((0.45, 1.30), 3.5, 1.0, fill=False, edgecolor="red", lw=2, ls="--"))
    ax.text(2.2, 1.15, f"P2 must choose one response for the belief, not for a known P1 card: {response}",
            ha="center", fontsize=8, color="red")

draw_belief_view(p2_card="Q", observed_action="Bet", bet_probs=DEFAULT_BET_PROBS)

In [ ]:
def bluffing_gui():
    if not WIDGETS_AVAILABLE:
        print("Install ipywidgets for sliders: pip install ipywidgets")
        return
    p_j = widgets.FloatSlider(value=0.20, min=0, max=1, step=0.05, description="P(Bet|J)")
    p_q = widgets.FloatSlider(value=0.55, min=0, max=1, step=0.05, description="P(Bet|Q)")
    p_k = widgets.FloatSlider(value=0.90, min=0, max=1, step=0.05, description="P(Bet|K)")
    p2 = widgets.Dropdown(options=CARDS, value="Q", description="P2 card")
    action = widgets.Dropdown(options=["Bet", "Check"], value="Bet", description="Observed")
    out = widgets.Output()
    def render(change=None):
        with out:
            clear_output(wait=True)
            probs = {"J": p_j.value, "Q": p_q.value, "K": p_k.value}
            ev = expected_value_p1_policy(probs)
            display(HTML(f"<b>Overall expected value for P1 policy:</b> {ev:.3f}"))
            draw_belief_view(p2.value, action.value, probs)
    for w in [p_j, p_q, p_k, p2, action]:
        w.observe(render, names="value")
    display(widgets.VBox([widgets.HBox([p_j, p_q, p_k]), widgets.HBox([p2, action]), out]))
    render()

bluffing_gui()

## Part B: Limitation 1 — exponential growth

Even before imperfect information, deterministic game trees grow as \(b^m\), where \(b\) is branching factor and \(m\) is depth. Alpha-beta can help, but it does not remove exponential growth.

In [ ]:
def plot_exponential_growth():
    depths = list(range(1, 11))
    fig, ax = plt.subplots(figsize=(9, 5))
    for b in [3, 7, 35]:
        minimax_nodes = [b**d for d in depths]
        alphabeta_best = [b**(d/2) for d in depths]
        ax.plot(depths, minimax_nodes, marker="o", label=f"minimax b={b}")
        ax.plot(depths, alphabeta_best, ls="--", label=f"ideal alpha-beta b={b}")
    ax.set_yscale("log")
    ax.set_xlabel("depth m")
    ax.set_ylabel("rough node count, log scale")
    ax.set_title("Game search grows exponentially")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(fontsize=8, ncol=2)
    plt.show()

plot_exponential_growth()

## Part C: Limitation 2 — horizon effect

A depth cutoff can prefer a delaying move because the real damage lies just beyond the search horizon. The toy game below has a safe move worth 0 and a delay chain that looks good at the cutoff but eventually reaches a disaster worth -10.

In [ ]:
def horizon_value(depth_limit, disaster_depth=5):
    def value_delay(remaining, depth):
        if remaining == 0:
            return -10
        if depth == 0:
            return 1  # heuristic says the delay position looks good
        return value_delay(remaining - 1, depth - 1)
    safe = 0
    delay = value_delay(disaster_depth, depth_limit)
    return {"depth_limit": depth_limit, "safe": safe, "delay": delay, "choice": "delay" if delay > safe else "safe"}

rows = [horizon_value(d, disaster_depth=5) for d in range(1, 8)]
for r in rows:
    print(r)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot([r["depth_limit"] for r in rows], [r["safe"] for r in rows], marker="o", label="safe move value")
ax.plot([r["depth_limit"] for r in rows], [r["delay"] for r in rows], marker="o", label="delay move value")
for r in rows:
    ax.text(r["depth_limit"], r["delay"], r["choice"], fontsize=8, ha="center", va="bottom")
ax.axhline(0, color="black", lw=0.8)
ax.set_xlabel("search depth limit")
ax.set_ylabel("backed-up value")
ax.set_title("Horizon effect: shallow search likes delaying tactics")
ax.grid(True, alpha=0.25)
ax.legend()
plt.show()

In [ ]:
def draw_horizon_tree(depth_limit=3, disaster_depth=5):
    fig, ax = plt.subplots(figsize=(11, 3.8))
    ax.axis("off")
    ax.set_xlim(-0.5, disaster_depth + 2)
    ax.set_ylim(-1.2, 1.2)
    ax.text(0, 0.8, "MAX", ha="center", fontsize=10, fontweight="bold")
    ax.scatter([0], [0], s=900, c="#d7e9ff", edgecolors="black")
    ax.text(0, 0, "root", ha="center", va="center")
    ax.plot([0, 1], [0, -0.7], color="#444444")
    ax.scatter([1], [-0.7], s=700, c="#b6e3a8", edgecolors="black")
    ax.text(1, -0.7, "safe\n0", ha="center", va="center")
    prev = (0, 0)
    for i in range(1, disaster_depth+1):
        x, y = i, 0.15
        ax.plot([prev[0], x], [prev[1], y], color="#444444")
        face = "#ffbf66" if i == depth_limit else "#fef3bd"
        label = f"delay {i}"
        if i == depth_limit:
            label += "\ncutoff EVAL=+1"
        if i == disaster_depth:
            label += "\ndisaster=-10"
            face = "#ffadad"
        ax.scatter([x], [y], s=800, c=face, edgecolors="black")
        ax.text(x, y, label, ha="center", va="center", fontsize=8)
        prev = (x, y)
    ax.set_title(f"Horizon visualization at depth limit {depth_limit}")
    plt.show()

draw_horizon_tree(depth_limit=3, disaster_depth=5)

## Part D: Limitation 3 — forward pruning can remove the best move

Forward pruning saves time by discarding moves that look bad before proving they are bad. This can be useful, but it can also prune the winning tactic.

In [ ]:
MOVES = [
    {"name": "obvious", "heuristic": 8, "true": 2},
    {"name": "solid", "heuristic": 7, "true": 1},
    {"name": "trap", "heuristic": 6, "true": -5},
    {"name": "hidden tactic", "heuristic": 1, "true": 10},
]

def beam_prune(moves, beam_width=2):
    ordered = sorted(moves, key=lambda m: m["heuristic"], reverse=True)
    kept = ordered[:beam_width]
    pruned = ordered[beam_width:]
    chosen_by_beam = max(kept, key=lambda m: m["true"])
    true_best = max(moves, key=lambda m: m["true"])
    return kept, pruned, chosen_by_beam, true_best

kept, pruned, chosen, true_best = beam_prune(MOVES, beam_width=2)
print("Kept:", kept)
print("Pruned:", pruned)
print("Beam chooses:", chosen)
print("True best:", true_best)

fig, ax = plt.subplots(figsize=(9, 5))
names = [m["name"] for m in MOVES]
heur = [m["heuristic"] for m in MOVES]
true = [m["true"] for m in MOVES]
x = list(range(len(MOVES)))
ax.bar([i-0.18 for i in x], heur, width=0.36, label="shallow heuristic", color="#8ecae6", edgecolor="black")
ax.bar([i+0.18 for i in x], true, width=0.36, label="true deep value", color="#b6e3a8", edgecolor="black")
for i, m in enumerate(MOVES):
    if m in pruned:
        ax.text(i, max(heur[i], true[i]) + 0.7, "PRUNED", ha="center", color="red", fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("score")
ax.set_title("Forward pruning risk: the hidden tactic looks bad at first")
ax.legend()
plt.show()

## Student practice

1. Change the bluffing probabilities. When should Player 2 call with Q after a bet?
2. Change the disaster depth. How deep must search go to avoid the horizon effect?
3. Change the beam width. When does the hidden tactic survive forward pruning?

In [ ]:
# Practice settings
practice_bet_probs = {"J": 0.05, "Q": 0.35, "K": 0.95}
practice_p2_card = "Q"
practice_belief = posterior_p1_card(practice_p2_card, "Bet", practice_bet_probs)
print("Belief after bet:", practice_belief)
print("P2 best response:", p2_best_response(practice_p2_card, practice_belief))
print("P1 policy EV:", round(expected_value_p1_policy(practice_bet_probs), 3))
draw_belief_view(practice_p2_card, "Bet", practice_bet_probs)

draw_horizon_tree(depth_limit=4, disaster_depth=6)
print("Beam width 3 result:", beam_prune(MOVES, beam_width=3))